## DATA CLEANSING NOTEBOOK

In [1]:
import os

In [2]:
curr_path = !pwd
curr_path = curr_path[0] + "/"
input_file = os.path.join(os.path.dirname(curr_path), '..', 'inputs', 'inventory_raw.csv')
input_file

'/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/notebooks/../inputs/inventory_raw.csv'

In [3]:
import re
import pandas as pd
import numpy as np
import ipaddress

from urllib.parse import urlparse

In [4]:
df_raw = pd.read_csv(input_file)
df_raw.head(5)

,source_row_id,ip,hostname,fqdn,mac,owner,device_type,site,notes
0,1,192.168.010.005,HOST01,NaN,AA-BB-CC-DD-EE-FF,priya (platform) priya@corp.example.com,server,BLR Campus,db host
1,2,10.0.1.300,host-02,host-02.local,11-22-33-44-55-66,ops,NaN,HQ Bldg 1,edge gw?
2,3,10.0.1,host03,NaN,aabb.ccdd.eeff,jane@corp.example.com,switch,HQ-BUILDING-1,NaN
3,4,10.0.1.1.2,printer-01,NaN,00:11:22:33:44:55,Facilities,printer,HQ,NaN
4,5,fe80::1%eth0,iot-cam01,NaN,00:aa:bb:cc:dd:ee,sec,iot,Lab-1,camera PoE on port 3


In [5]:
df_raw.columns

Index(['source_row_id', 'ip', 'hostname', 'fqdn', 'mac', 'owner',
       'device_type', 'site', 'notes'],
      dtype='object')

### IP

IP VERSION

In [6]:
# TODO: IMPLEMENT IP_VERSION CLASSIFICATION
# def detect_ip_version(ip_str):
#     if pd.isna(ip_str):
#         return "invalid"
#     s = str(ip_str).strip()

#     if s == "":
#         return "invalid"
    
#     colon_count = s.count(':')
#     dot_count = s.count('.')

#     # Ipv6 indicators
#     if colon_count >= 2:
#         try:
#             ipaddress.IPv6Address(s)
#             return "6"
#         except ipaddress.AddressValueError as e:
#             return "invalid"
        
#     elif dot_count == 3:
#         try:
#             ipaddress.IPv4Address(s)
#             return "4"
#         except ipaddress.AddressValueError as e:
#             return "invalid"
        
#     else:
#         return "invalid"

VALIDATION & NORMALIZATION

In [7]:
def traceability_metadata(field: str, from_value, to_value, reason: str):
    return {
        "field": field,
        "from": from_value,
        "to": to_value,
        "reason": reason
    }

In [8]:
def ipv4_validate_and_normalize(ip_str):
    FIELD="ip"
    FROM=ip_str

    if pd.isna(ip_str):
        final_ip = None
        return (False, final_ip, traceability_metadata(FIELD, FROM, final_ip, "missing"))
    s = str(ip_str).strip()
    if ':' in s:
        final_ip = None
        return (False, final_ip, traceability_metadata(FIELD, FROM, final_ip, "ipv6_or_non_ipv4"))
    parts = s.split(".")
    if len(parts) != 4:
        final_ip = None
        return (False, final_ip, traceability_metadata(FIELD, FROM, final_ip, "wrong_octet_count"))
    
    canonical_parts = []
    for p in parts:
        if p == '':
            final_ip = None
            return (False, final_ip, traceability_metadata(FIELD, FROM, final_ip, "empty_octet"))
        if not (p.lstrip("+").isdigit() and not p.startswith("-")):
            final_ip = None
            return (False, final_ip, traceability_metadata(FIELD, FROM, final_ip, "non_numeric_or_negative"))
        try:
            v = int(p, 10)
        except ValueError:
            final_ip = None
            return (False, final_ip, traceability_metadata(FIELD, FROM, final_ip, "non_decimal_format"))
        if v < 0 or v > 255:
            final_ip = None
            return (False, final_ip, traceability_metadata(FIELD, FROM, final_ip, "octet_out_of_range"))
        canonical_parts.append(str(v))
    canonical = ".".join(canonical_parts)

    
    return (True, canonical, traceability_metadata(FIELD, FROM, canonical, "ok"))

IP_TYPE

In [9]:
def classify_ipv4_type(ip):
    if pd.isna(ip) or ip is None:
        return "invalid"
    
    try:
        octets = list(map(int, ip.split('.')))

        # Checking private ranges
        if octets[0] == 10:
            return "private_rfc1918"
        elif octets[0] == 172 and 16 <= octets[1] <= 31:
            return "private_rfc1918"
        elif octets[0] == 192 and octets[1] == 168:
            return "private_rfc1918"
        
        # Other ranges
        elif octets[0] == 169 and octets[1] == 254:
            return "link_local_apipa"
        elif octets[0] == 127:
            return "loopback"
        
        return "public_or_other"
    except (ValueError, AttributeError, IndexError):
        return "invalid_format"

SUBNET_CIDR

In [10]:
def default_subnet(ip, ip_type=None):
    if pd.isna(ip) or ip is None:
        return ""
    
    try:
        octets = list(map(int, ip.split('.')))

        # Get Ip type, if not provided
        if ip_type is None:
            ip_type = classify_ipv4_type_pandas(ip)

        # Subnet strategies
        if ip_type == 'private_rfc1918':
            if octets[0] == 10:
                return f"10.0.0.0/8"
            elif octets[0] == 172:
                return f"172.{octets[1]}.0.0/16"
            elif octets[0] == 192 and octets[1] == 168:
                return f"192.168.{octets[2]}.0/24"
            
        elif ip_type == "link_local_apipa":
            return "169.254.0.0/16"

        elif ip_type == "loopback":
            return "127.0.0.0/8"
        
        elif ip_type == "public_or_other":
            # Assumign /24 for public IPs
            return f"{octets[0]}.{octets[1]}.{octets[2]}.0/24"
        
        return ""
    
    except (ValueError, AttributeError, IndexError):
        return ""

In [11]:
df_ipv4 = pd.DataFrame()
df_ipv4[['ip_valid', 'ip_canonical', 'tr_metadata']] = df_raw['ip'].apply(
    lambda x: pd.Series(ipv4_validate_and_normalize(x))
)
df_ipv4['ip_type'] = df_ipv4['ip_canonical'].apply(
    lambda x: pd.Series(classify_ipv4_type(x))
)
df_ipv4['subnet_cidr'] = df_ipv4.apply(
    lambda x: default_subnet(x['ip_canonical'], x['ip_type'])
    if x['ip_valid'] else "",
    axis=1
)
df_ipv4

,ip_valid,ip_canonical,tr_metadata,ip_type,subnet_cidr
0,True,192.168.10.5,"{'field': 'ip', 'from': '192.168.010.005', 'to...",private_rfc1918,192.168.10.0/24
1,False,None,"{'field': 'ip', 'from': '10.0.1.300', 'to': No...",invalid,
2,False,None,"{'field': 'ip', 'from': '10.0.1', 'to': None, ...",invalid,
3,False,None,"{'field': 'ip', 'from': '10.0.1.1.2', 'to': No...",invalid,
4,False,None,"{'field': 'ip', 'from': 'fe80::1%eth0', 'to': ...",invalid,
5,True,127.0.0.1,"{'field': 'ip', 'from': '127.0.0.1', 'to': '12...",loopback,127.0.0.0/8
6,True,169.254.10.20,"{'field': 'ip', 'from': '169.254.10.20', 'to':...",link_local_apipa,169.254.0.0/16
7,True,10.10.10.10,"{'field': 'ip', 'from': ' 10.10.10.10 ', 'to...",private_rfc1918,10.0.0.0/8
8,False,None,"{'field': 'ip', 'from': 'abc.def.ghi.jkl', 'to...",invalid,
9,False,None,"{'field': 'ip', 'from': '192.168.1.-1', 'to': ...",invalid,


### HOSTNAME

VALIDATION & NORMALIZATION

In [12]:
def validate_hostname(hostname_str):
    """Validating hostname according to RFC1123"""

    FIELD='hostname'
    FROM=hostname_str

    if pd.isna(hostname_str):
        final_hostname = None
        return (False, None, traceability_metadata(FIELD, FROM, final_hostname, "missing"))
    
    s = str(hostname_str).strip()

    final_hostname = s

    if s == "":
        final_hostname = None
        return (False, final_hostname, traceability_metadata(FIELD, FROM, final_hostname, "empty_string"))
    
    if len(s) > 63:
        return (False, final_hostname, traceability_metadata(FIELD, FROM, final_hostname, "too_long"))
    
    # Checking periods
    if '.' in s:
        return (False, final_hostname, traceability_metadata(FIELD, FROM, final_hostname, "contains_periods"))
    
    # Character validation - RFC 1123
    if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', s):
        return (False, final_hostname, traceability_metadata(FIELD, FROM, final_hostname, "invalid_characters"))
    
    # Cannot start or end with hyphen
    if s.startswith('-') or s.endswith('-'):
        return (False, final_hostname, traceability_metadata(FIELD, FROM, final_hostname, "hyphen_at_edge"))
    
    # Normalize to lowercase for consistency
    canonical = s.lower()

    return (True, canonical, traceability_metadata(FIELD, FROM, canonical, "ok"))
    

In [13]:
df_hostname = pd.DataFrame()
df_hostname['hostname'] = df_raw['hostname']
df_hostname[['hostname_valid', 'hostname_canonical', 'tr_metadata']] = df_hostname['hostname'].apply(
    lambda x: pd.Series(validate_hostname(x))
)
df_hostname

,hostname,hostname_valid,hostname_canonical,tr_metadata
0,HOST01,True,host01,"{'field': 'hostname', 'from': 'HOST01', 'to': ..."
1,host-02,True,host-02,"{'field': 'hostname', 'from': 'host-02', 'to':..."
2,host03,True,host03,"{'field': 'hostname', 'from': 'host03', 'to': ..."
3,printer-01,True,printer-01,"{'field': 'hostname', 'from': 'printer-01', 't..."
4,iot-cam01,True,iot-cam01,"{'field': 'hostname', 'from': 'iot-cam01', 'to..."
5,local-test,True,local-test,"{'field': 'hostname', 'from': 'local-test', 't..."
6,host-apipa,True,host-apipa,"{'field': 'hostname', 'from': 'host-apipa', 't..."
7,srv-10,True,srv-10,"{'field': 'hostname', 'from': 'srv-10', 'to': ..."
8,badhost,True,badhost,"{'field': 'hostname', 'from': 'badhost', 'to':..."
9,neg,True,neg,"{'field': 'hostname', 'from': 'neg', 'to': 'ne..."


### SITE

NORMALIZATION

In [14]:
def normalize_site(site: str) -> str:
    """
    Normalize a site name string and return a standardized form.
    
    Examples:
      'BLR Campus'    -> 'blr-campus'
      'HQ Bldg 1'     -> 'hq-bldg-1'
      'HQ-BUILDING-1' -> 'hq-bldg-1'
      'Lab-1'         -> 'lab-1'
      NaN or None     -> 'unknown'
    """
    FIELD='site'
    FROM=site

    final_site = 'unknown'
    if pd.isna(site) or site is None:
        return (final_site, traceability_metadata(FIELD, FROM, final_site, "missing"))
    
    s = str(site).strip().lower()
    if s == "":
        return (final_site, traceability_metadata(FIELD, FROM, final_site, "empty_string"))
    
    # Replace underscores, multiple spaces, and commas with hyphens
    s = re.sub(r"[\s,_]+", "-", s)
    
    # Common standardization: building → bldg, campus → campus (unchanged)
    s = s.replace("building", "bldg")
    
    # Remove duplicate hyphens
    s = re.sub(r"-{2,}", "-", s)
    
    # Strip trailing or leading hyphens
    s = s.strip("-")
    
    # Handle very generic cases
    if s in {"n/a", "na", "none", "null", "unknown"}:
        return (final_site, traceability_metadata(FIELD, FROM, final_site, "missing"))
    
    return (s, traceability_metadata(FIELD, FROM, s, "normalized_site"))

In [15]:
df_site = pd.DataFrame()

df_site['original_site'] = df_raw['site']
df_site[['site_normalized', 'tr_metadata']] = df_raw['site'].apply(
    lambda x: pd.Series(normalize_site(x))
).apply(pd.Series)
df_site

,original_site,site_normalized,tr_metadata
0,BLR Campus,blr-campus,"{'field': 'site', 'from': 'BLR Campus', 'to': ..."
1,HQ Bldg 1,hq-bldg-1,"{'field': 'site', 'from': 'HQ Bldg 1', 'to': '..."
2,HQ-BUILDING-1,hq-bldg-1,"{'field': 'site', 'from': 'HQ-BUILDING-1', 'to..."
3,HQ,hq,"{'field': 'site', 'from': 'HQ', 'to': 'hq', 'r..."
4,Lab-1,lab-1,"{'field': 'site', 'from': 'Lab-1', 'to': 'lab-..."
5,NaN,unknown,"{'field': 'site', 'from': nan, 'to': 'unknown'..."
6,NaN,unknown,"{'field': 'site', 'from': nan, 'to': 'unknown'..."
7,BLR campus,blr-campus,"{'field': 'site', 'from': 'BLR campus', 'to': ..."
8,NaN,unknown,"{'field': 'site', 'from': nan, 'to': 'unknown'..."
9,NaN,unknown,"{'field': 'site', 'from': nan, 'to': 'unknown'..."


### FQDN

VALDIATION & NORMALIZATION

In [16]:
def validate_fqdn(fqdn_str, hostname_part=None, site_part=None):
    """
    Validates FQDN according to RFC 1035 and checks consistency with hostname/site.
    Returns a tuple: (is_valid: bool, normalized_value: str, error_code: str, consistency: str)
    """
    FIELD='fqdn'
    FROM=fqdn_str

    final_fqdn = None
    if pd.isna(fqdn_str) or fqdn_str is None:
        return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, "missing"), "inconsistent")
    
    s = str(fqdn_str).strip()

    if s == "":
        return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, "empty_string"), "inconsistent")
    
    final_fqdn = s
    # Remove trailing dot if present (optional in some systems)
    if s.endswith('.'):
        s = s[:-1]
    
    # Check overall length (RFC 1035: max 255 chars including dots)
    if len(s) > 255:
        return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, "too_long"), "inconsistent")
    
    # Split into labels
    labels = s.split('.')
    
    # Must have at least 2 labels (hostname + domain)
    if len(labels) < 2:
        return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, "too_few_labels"), "inconsistent")
    
    # Validate each label
    for i, label in enumerate(labels):
        # Check label length (max 63 chars per RFC 1035)
        if len(label) > 63:
            return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, f"label_{i}_too_long"), "inconsistent")
        
        # Check for empty labels (consecutive periods)
        if len(label) == 0:
            return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, "empty_label"), "inconsistent")
        
        # First and last label have special rules
        if i == 0:
            # First label is essentially the hostname - apply hostname rules
            if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', label):
                return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, "invalid_hostname_label"), "inconsistent")
        else:
            # Other labels (domain parts) can start/end with digits
            # But still no leading/trailing hyphens
            if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', label):
                # Allow digits at start/end for domain labels (like "2ndfloor" or "lab1")
                if not re.match(r'^[a-zA-Z0-9-]+$', label):
                    return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, f"invalid_domain_label{i}"), "inconsistent")
        
        # Check for leading/trailing hyphens in all labels
        if label.startswith('-') or label.endswith('-'):
            return (False, final_fqdn, traceability_metadata(FIELD, FROM, final_fqdn, f"label_{i}_hyphen_edge"), "inconsistent")
    
    # Normalize to lowercase
    canonical = s.lower()
    
    # Check consistency with hostname and site if provided
    consistency_status = "unknown"
    if hostname_part is not None and site_part is not None:
        expected_fqdn = f"{hostname_part}.{site_part}".lower()
        if canonical == expected_fqdn:
            consistency_status = "consistent"
        else:
            consistency_status = "inconsistent"

    
    return (True, canonical, traceability_metadata(FIELD, FROM, canonical, "valid"), consistency_status)

In [17]:

def generate_reverse_ptr(ip):
    if pd.isna(ip) or ip is None:
        return None

    s_ip = str(ip).strip()
    if s_ip == "":
        return None

    try:
        ip_obj = ipaddress.ip_address(s_ip)
    except ValueError:
        return None

    reverse_key = ip_obj.reverse_pointer.lower().rstrip('.') 

    return reverse_key

In [18]:
df_fqdn = pd.DataFrame()
df_fqdn['fqdn'] = df_raw['fqdn']
df_fqdn['hostname_canonical'] = df_hostname['hostname_canonical']
df_fqdn['site'] = df_site['site_normalized']
df_fqdn[['reverse_ptr']] = df_ipv4['ip_canonical'].apply(
    lambda x: pd.Series(generate_reverse_ptr(x))
)
df_fqdn[['fqdn_valid', 'fqdn_canonical', 'tr_metadata', 'fqdn_consistent']] = (
    df_fqdn.apply(
        lambda x: validate_fqdn(x['fqdn'], x['hostname_canonical'], x['site']),
        axis='columns'
    ).apply(pd.Series)
)
df_fqdn


,fqdn,hostname_canonical,site,reverse_ptr,fqdn_valid,fqdn_canonical,tr_metadata,fqdn_consistent
0,NaN,host01,blr-campus,5.10.168.192.in-addr.arpa,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent
1,host-02.local,host-02,hq-bldg-1,NaN,True,host-02.local,"{'field': 'fqdn', 'from': 'host-02.local', 'to...",inconsistent
2,NaN,host03,hq-bldg-1,NaN,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent
3,NaN,printer-01,hq,NaN,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent
4,NaN,iot-cam01,lab-1,NaN,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent
5,NaN,local-test,unknown,1.0.0.127.in-addr.arpa,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent
6,NaN,host-apipa,unknown,20.10.254.169.in-addr.arpa,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent
7,NaN,srv-10,blr-campus,10.10.10.10.in-addr.arpa,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent
8,NaN,badhost,unknown,NaN,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent
9,NaN,neg,unknown,NaN,False,None,"{'field': 'fqdn', 'from': nan, 'to': None, 're...",inconsistent


### MAC

NORMALIZATION AND VALIDATION

In [19]:
def validate_mac(mac_str):
    """
    Validates a MAC address string.
    Returns a tuple: (is_valid: bool, normalized_value: str | None, error_code: str)
    
    Rules:
      - Must not be missing or empty
      - Must match one of the common formats (colon, hyphen, or dot separated)
      - Normalized output: colon-separated lowercase (e.g., 'aa:bb:cc:dd:ee:ff')
    """

    FIELD='mac'
    FROM=mac_str
    
    final_mac = None
    if pd.isna(mac_str) or mac_str is None:
        return (False, final_mac, traceability_metadata(FIELD, FROM, final_mac, "missing"))
    
    s = str(mac_str).strip()
    if s == "":
        return (False, final_mac, traceability_metadata(FIELD, FROM, final_mac, "empty_string"))

    # Define regex patterns for common formats
    patterns = [
        r"^([0-9A-Fa-f]{2}[:-]){5}([0-9A-Fa-f]{2})$",   # AA:BB:CC:DD:EE:FF or AA-BB-CC-DD-EE-FF
        r"^([0-9A-Fa-f]{4}\.){2}([0-9A-Fa-f]{4})$",     # AAAA.BBBB.CCCC
    ]
    
    # Check match
    if not any(re.match(p, s) for p in patterns):
        return (False, final_mac, traceability_metadata(FIELD, FROM, final_mac, "invalid_format"))
    
    # Normalize to colon-separated lowercase format
    s = s.lower().replace('-', ':')
    if '.' in s:
        # Cisco-style -> flatten and regroup
        s = s.replace('.', '')
        s = ':'.join([s[i:i+2] for i in range(0, 12, 2)])

    
    return (True, s, traceability_metadata(FIELD, FROM, s, "valid"))

In [20]:
df_mac = pd.DataFrame()
df_mac['mac'] = df_raw['mac']
df_mac[['mac_valid', 'mac_canonical', 'tr_metadata']] = df_mac['mac'].apply(
    lambda x: pd.Series(validate_mac(x))
)
df_mac

,mac,mac_valid,mac_canonical,tr_metadata
0,AA-BB-CC-DD-EE-FF,True,aa:bb:cc:dd:ee:ff,"{'field': 'mac', 'from': 'AA-BB-CC-DD-EE-FF', ..."
1,11-22-33-44-55-66,True,11:22:33:44:55:66,"{'field': 'mac', 'from': '11-22-33-44-55-66', ..."
2,aabb.ccdd.eeff,True,aa:bb:cc:dd:ee:ff,"{'field': 'mac', 'from': 'aabb.ccdd.eeff', 'to..."
3,00:11:22:33:44:55,True,00:11:22:33:44:55,"{'field': 'mac', 'from': '00:11:22:33:44:55', ..."
4,00:aa:bb:cc:dd:ee,True,00:aa:bb:cc:dd:ee,"{'field': 'mac', 'from': '00:aa:bb:cc:dd:ee', ..."
5,NaN,False,None,"{'field': 'mac', 'from': nan, 'to': None, 'rea..."
6,NaN,False,None,"{'field': 'mac', 'from': nan, 'to': None, 'rea..."
7,NaN,False,None,"{'field': 'mac', 'from': nan, 'to': None, 'rea..."
8,NaN,False,None,"{'field': 'mac', 'from': nan, 'to': None, 'rea..."
9,NaN,False,None,"{'field': 'mac', 'from': nan, 'to': None, 'rea..."


### OWNER

PARSING

In [29]:
def parse_owner(owner_str):
    """
    Parses a single owner string into (owner, owner_email, owner_team).
    
    Returns:
        tuple: (owner, owner_email, owner_team)
    """
    FIELD='owner'
    FROM=owner_str

    final_owner=None
    if pd.isna(owner_str) or not str(owner_str).strip():
        return (None, None, None, traceability_metadata(FIELD, FROM, final_owner, "missing"))
    
    s = str(owner_str).strip()
    
    # Regex for email detection
    email_pattern = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")
    
    # Extract email if present
    email_match = email_pattern.search(s)
    email = email_match.group(0) if email_match else None
    s_no_email = email_pattern.sub("", s).strip()
    
    # Extract team (inside parentheses)
    team_match = re.search(r"\((.*?)\)", s_no_email)
    team = team_match.group(1).strip() if team_match else None
    s_no_paren = re.sub(r"\(.*?\)", "", s_no_email).strip()
    
    # Remaining part is likely the owner name or alias
    owner = s_no_paren.lower() if s_no_paren else None
    
    return (owner, email, team, traceability_metadata(FIELD, FROM, owner, "owner_valid_and_normalized"))

In [30]:
# TODO: MIGHT HAVE TO PASS THIS ONE THROUGH THE LLM
df_owner = pd.DataFrame()
df_owner['original_owner'] = df_raw['owner']
df_owner[['owner', 'owner_email', 'owner_team', 'tr_metadata']] = df_raw['owner'].apply(
    lambda x: parse_owner(x)
).apply(pd.Series)
df_owner

,original_owner,owner,owner_email,owner_team,tr_metadata
0,priya (platform) priya@corp.example.com,priya,priya@corp.example.com,platform,"{'field': 'owner', 'from': 'priya (platform) p..."
1,ops,ops,None,None,"{'field': 'owner', 'from': 'ops', 'to': 'ops',..."
2,jane@corp.example.com,None,jane@corp.example.com,None,"{'field': 'owner', 'from': 'jane@corp.example...."
3,Facilities,facilities,None,None,"{'field': 'owner', 'from': 'Facilities', 'to':..."
4,sec,sec,None,None,"{'field': 'owner', 'from': 'sec', 'to': 'sec',..."
5,NaN,None,None,None,"{'field': 'owner', 'from': nan, 'to': None, 'r..."
6,NaN,None,None,None,"{'field': 'owner', 'from': nan, 'to': None, 'r..."
7,platform,platform,None,None,"{'field': 'owner', 'from': 'platform', 'to': '..."
8,NaN,None,None,None,"{'field': 'owner', 'from': nan, 'to': None, 'r..."
9,NaN,None,None,None,"{'field': 'owner', 'from': nan, 'to': None, 'r..."


### DEVICE_TYPE

CLASSIFICATION

In [23]:
def normalize_device_type(device_type: str):
    """
    Normalize a device_type string and return a tuple:
      (normalized_device_type: str, device_type_confidence: int)
      
    Rules:
      - 100 → exact match to known canonical type
      - 90  → fuzzy or alias match (e.g. abbreviation, keyword)
      - 0   → unrecognized or missing
    """

    # Canonical device types
    canonical_types = {
        "switch", "router", "firewall", "server", "printer",
        "wireless_ap", "wireless_controller", "load_balancer",
        "storage", "ups", "ip_phone", "camera", "unknown"
    }

    FIELD='device_type'
    FROM=device_type

    final_device_type = 'unknown'

    # Strong / exact aliases → canonical
    strong_map = {
        "switch": "switch",
        "router": "router",
        "firewall": "firewall",
        "server": "server",
        "printer": "printer",
        "access_point": "wireless_ap",
        "wireless_ap": "wireless_ap",
        "controller": "wireless_controller",
        "load_balancer": "load_balancer",
        "storage": "storage",
        "ups": "ups",
        "phone": "ip_phone",
        "camera": "camera"
    }

    # Fuzzy patterns for common abbreviations or variations
    fuzzy_patterns = [
        (r"\bsw\b|switch", "switch"),
        (r"\brtr\b|router", "router"),
        (r"\bfw\b|firewall|asa|pa\d+", "firewall"),
        (r"\bap\b|access[_-]?point", "wireless_ap"),
        (r"wlc|controller", "wireless_controller"),
        (r"f5|ltm|lb|load[_-]?balancer", "load_balancer"),
        (r"server|vm|esxi", "server"),
        (r"printer|print", "printer"),
        (r"ups", "ups"),
        (r"voip|phone", "ip_phone"),
        (r"camera|cam", "camera"),
        (r"nas|san|storage", "storage"),
    ]

    # --- Clean input ---
    if pd.isna(device_type) or device_type is None:
        return ("unknown", 0 , traceability_metadata(FIELD, FROM, final_device_type, "missing"))

    s = str(device_type).strip().lower()
    s = re.sub(r"[,_;/]", " ", s)
    s = re.sub(r"\s+", " ", s)
    if s == "":
        return ("unknown", 0, traceability_metadata(FIELD, FROM, final_device_type, "empty_string"))

    # --- Exact match ---
    if s in strong_map:
        final_device_type = strong_map[s]
        return (strong_map[s], 100, traceability_metadata(FIELD, FROM, final_device_type, "exact_match"))
    if s in canonical_types:
        final_device_type = s
        return (s, 100, traceability_metadata(FIELD, FROM, final_device_type, "exact_match"))

    # --- Fuzzy match ---
    for pattern, normalized in fuzzy_patterns:
        final_device_type = normalized
        if re.search(pattern, s):
            return (normalized, 90, traceability_metadata(FIELD, FROM, final_device_type, "fuzzy_match"))
        

    # --- No match ---
    return ("unknown", 0, traceability_metadata(FIELD, FROM, final_device_type, "no_match"))

In [24]:
df_dt = pd.DataFrame()
df_dt['original_dt'] = df_raw['device_type']
df_dt[['device_type', 'device_type_confidence', 'tr_metadata']] = df_raw['device_type'].apply(
    lambda x: normalize_device_type(x)
).apply(pd.Series)

df_dt

,original_dt,device_type,device_type_confidence,tr_metadata
0,server,server,100,"{'field': 'device_type', 'from': 'server', 'to..."
1,NaN,unknown,0,"{'field': 'device_type', 'from': nan, 'to': 'u..."
2,switch,switch,100,"{'field': 'device_type', 'from': 'switch', 'to..."
3,printer,printer,100,"{'field': 'device_type', 'from': 'printer', 't..."
4,iot,unknown,0,"{'field': 'device_type', 'from': 'iot', 'to': ..."
5,NaN,unknown,0,"{'field': 'device_type', 'from': nan, 'to': 'u..."
6,NaN,unknown,0,"{'field': 'device_type', 'from': nan, 'to': 'u..."
7,server,server,100,"{'field': 'device_type', 'from': 'server', 'to..."
8,NaN,unknown,0,"{'field': 'device_type', 'from': nan, 'to': 'u..."
9,NaN,unknown,0,"{'field': 'device_type', 'from': nan, 'to': 'u..."


### NORMALIZATION STEPS

In [25]:
def append_metadata(row):
    """Inputs: (row) row of df_normalization_steps
    Returns: (metdata_steps) list including all steps in each row"""
    metadata_steps = []

    for i in range(1, len(row)):
        metadata_steps.append(row[i])

    return metadata_steps

In [26]:
df_normalization_steps = pd.DataFrame({
    "source_row_id": df_raw['source_row_id'],
    "ip_traceability_metadata": df_ipv4['tr_metadata'],
    "hostname_traceability_metadata": df_hostname['tr_metadata'],
    "fqdn_traceability_metadata": df_fqdn['tr_metadata'],
    "mac_traceability_metadata": df_mac['tr_metadata'],
    "owner_traceability_metadata": df_owner['tr_metadata'],
    "device_type_traceability_metadata": df_dt['tr_metadata'],
    "site_traceability_metadata": df_site['tr_metadata']
})

df_normalization_steps['normalization_steps'] = df_normalization_steps.apply(
    lambda x: append_metadata(x),
    axis=1
)

df_normalization_steps.head(5)

/var/folders/yn/9tc_s2bs11ncssnxtt95x_l00000gn/T/ipykernel_14560/2776079095.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  metadata_steps.append(row[i])


,source_row_id,ip_traceability_metadata,hostname_traceability_metadata,fqdn_traceability_metadata,mac_traceability_metadata,owner_traceability_metadata,device_type_traceability_metadata,site_traceability_metadata,normalization_steps
0,1,"{'field': 'ip', 'from': '192.168.010.005', 'to...","{'field': 'hostname', 'from': 'HOST01', 'to': ...","{'field': 'fqdn', 'from': nan, 'to': None, 're...","{'field': 'mac', 'from': 'AA-BB-CC-DD-EE-FF', ...","{'field': 'owner', 'from': 'priya (platform) p...","{'field': 'device_type', 'from': 'server', 'to...","{'field': 'site', 'from': 'BLR Campus', 'to': ...","[{'field': 'ip', 'from': '192.168.010.005', 't..."
1,2,"{'field': 'ip', 'from': '10.0.1.300', 'to': No...","{'field': 'hostname', 'from': 'host-02', 'to':...","{'field': 'fqdn', 'from': 'host-02.local', 'to...","{'field': 'mac', 'from': '11-22-33-44-55-66', ...","{'field': 'owner', 'from': 'ops', 'to': 'ops',...","{'field': 'device_type', 'from': nan, 'to': 'u...","{'field': 'site', 'from': 'HQ Bldg 1', 'to': '...","[{'field': 'ip', 'from': '10.0.1.300', 'to': N..."
2,3,"{'field': 'ip', 'from': '10.0.1', 'to': None, ...","{'field': 'hostname', 'from': 'host03', 'to': ...","{'field': 'fqdn', 'from': nan, 'to': None, 're...","{'field': 'mac', 'from': 'aabb.ccdd.eeff', 'to...","{'field': 'owner', 'from': 'jane@corp.example....","{'field': 'device_type', 'from': 'switch', 'to...","{'field': 'site', 'from': 'HQ-BUILDING-1', 'to...","[{'field': 'ip', 'from': '10.0.1', 'to': None,..."
3,4,"{'field': 'ip', 'from': '10.0.1.1.2', 'to': No...","{'field': 'hostname', 'from': 'printer-01', 't...","{'field': 'fqdn', 'from': nan, 'to': None, 're...","{'field': 'mac', 'from': '00:11:22:33:44:55', ...","{'field': 'owner', 'from': 'Facilities', 'to':...","{'field': 'device_type', 'from': 'printer', 't...","{'field': 'site', 'from': 'HQ', 'to': 'hq', 'r...","[{'field': 'ip', 'from': '10.0.1.1.2', 'to': N..."
4,5,"{'field': 'ip', 'from': 'fe80::1%eth0', 'to': ...","{'field': 'hostname', 'from': 'iot-cam01', 'to...","{'field': 'fqdn', 'from': nan, 'to': None, 're...","{'field': 'mac', 'from': '00:aa:bb:cc:dd:ee', ...","{'field': 'owner', 'from': 'sec', 'to': 'sec',...","{'field': 'device_type', 'from': 'iot', 'to': ...","{'field': 'site', 'from': 'Lab-1', 'to': 'lab-...","[{'field': 'ip', 'from': 'fe80::1%eth0', 'to':..."


In [27]:
df_normalization_steps['normalization_steps'].iloc[1]

[{'field': 'ip',
  'from': '10.0.1.300',
  'to': None,
  'reason': 'octet_out_of_range'},
 {'field': 'hostname', 'from': 'host-02', 'to': 'host-02', 'reason': 'ok'},
 {'field': 'fqdn',
  'from': 'host-02.local',
  'to': 'host-02.local',
  'reason': 'valid'},
 {'field': 'mac',
  'from': '11-22-33-44-55-66',
  'to': '11:22:33:44:55:66',
  'reason': 'valid'},
 {'field': 'owner',
  'from': 'ops',
  'to': 'ops',
  'reason': 'owner_valid_and_normalized'},
 {'field': 'device_type', 'from': nan, 'to': 'unknown', 'reason': 'missing'},
 {'field': 'site',
  'from': 'HQ Bldg 1',
  'to': 'hq-bldg-1',
  'reason': 'normalized_site'}]

### PUTTING IT ALL TOGETHER

In [35]:
df_final = pd.DataFrame()
# IP 
df_final['ip'] = df_ipv4['ip_canonical']
df_final['ip_valid'] = df_ipv4['ip_valid']
df_final['ip_type'] = df_ipv4['ip_type']
df_final['subnet_cidr'] = df_ipv4['subnet_cidr']

# Hostname
df_final['hostname'] = df_hostname['hostname_canonical']
df_final['hostname_valid'] = df_hostname['hostname_valid']

# FQDN
df_final['fqdn'] = df_fqdn['fqdn_canonical']
df_final['fqdn_consistent'] = df_fqdn['fqdn_consistent']
df_final['reverse_ptr'] = df_fqdn['reverse_ptr']

# MAC
df_final['mac'] = df_mac['mac_canonical']
df_final['mac_valid'] = df_mac['mac_valid']

# OWNER
df_final['owner'] = df_owner['owner']
df_final['owner_email'] = df_owner['owner_email']
df_final['owner_team'] = df_owner['owner_team']

# DEVICE_TYPE
df_final['device_type'] = df_dt['device_type']
df_final['device_type_confidence'] = df_dt['device_type_confidence']

# SITE
df_final['site'] = df_raw['site']
df_final['site_normalized'] = df_site['site_normalized']

# NORMALIZATION_STEPS
df_final['source_row_id'] = df_normalization_steps['source_row_id']
df_final['normalization_steps'] = df_normalization_steps['normalization_steps']

df_final

,ip,ip_valid,ip_type,subnet_cidr,hostname,hostname_valid,fqdn,fqdn_consistent,reverse_ptr,mac,mac_valid,owner,owner_email,owner_team,device_type,device_type_confidence,site,site_normalized,source_row_id,normalization_steps
0,192.168.10.5,True,private_rfc1918,192.168.10.0/24,host01,True,None,inconsistent,5.10.168.192.in-addr.arpa,aa:bb:cc:dd:ee:ff,True,priya,priya@corp.example.com,platform,server,100,BLR Campus,blr-campus,1,"[{'field': 'ip', 'from': '192.168.010.005', 't..."
1,None,False,invalid,,host-02,True,host-02.local,inconsistent,NaN,11:22:33:44:55:66,True,ops,None,None,unknown,0,HQ Bldg 1,hq-bldg-1,2,"[{'field': 'ip', 'from': '10.0.1.300', 'to': N..."
2,None,False,invalid,,host03,True,None,inconsistent,NaN,aa:bb:cc:dd:ee:ff,True,None,jane@corp.example.com,None,switch,100,HQ-BUILDING-1,hq-bldg-1,3,"[{'field': 'ip', 'from': '10.0.1', 'to': None,..."
3,None,False,invalid,,printer-01,True,None,inconsistent,NaN,00:11:22:33:44:55,True,facilities,None,None,printer,100,HQ,hq,4,"[{'field': 'ip', 'from': '10.0.1.1.2', 'to': N..."
4,None,False,invalid,,iot-cam01,True,None,inconsistent,NaN,00:aa:bb:cc:dd:ee,True,sec,None,None,unknown,0,Lab-1,lab-1,5,"[{'field': 'ip', 'from': 'fe80::1%eth0', 'to':..."
5,127.0.0.1,True,loopback,127.0.0.0/8,local-test,True,None,inconsistent,1.0.0.127.in-addr.arpa,None,False,None,None,None,unknown,0,NaN,unknown,6,"[{'field': 'ip', 'from': '127.0.0.1', 'to': '1..."
6,169.254.10.20,True,link_local_apipa,169.254.0.0/16,host-apipa,True,None,inconsistent,20.10.254.169.in-addr.arpa,None,False,None,None,None,unknown,0,NaN,unknown,7,"[{'field': 'ip', 'from': '169.254.10.20', 'to'..."
7,10.10.10.10,True,private_rfc1918,10.0.0.0/8,srv-10,True,None,inconsistent,10.10.10.10.in-addr.arpa,None,False,platform,None,None,server,100,BLR campus,blr-campus,8,"[{'field': 'ip', 'from': ' 10.10.10.10 ', 't..."
8,None,False,invalid,,badhost,True,None,inconsistent,NaN,None,False,None,None,None,unknown,0,NaN,unknown,9,"[{'field': 'ip', 'from': 'abc.def.ghi.jkl', 't..."
9,None,False,invalid,,neg,True,None,inconsistent,NaN,None,False,None,None,None,unknown,0,NaN,unknown,10,"[{'field': 'ip', 'from': '192.168.1.-1', 'to':..."
